# 02 · Feed-forward networks with entity embeddings
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/actuarialsociety/deep-actuarial-science/blob/main/notebooks/02_fnn_embeddings.ipynb)

$\lambda(x)=\exp\{w_0+\langle w, z^{(d:1)}(x)\rangle\}$ with 2-D embeddings for brand and region. Full data: about 5-10 minutes on a CPU.

In [ ]:
# Setup: works in Colab, the dev container and a local clone.
import os, sys
if "google.colab" in sys.modules and not os.path.exists("/content/deep-actuarial-science"):
    !git clone -q https://github.com/actuarialsociety/deep-actuarial-science /content/deep-actuarial-science
    !pip install -q -e /content/deep-actuarial-science
if "google.colab" in sys.modules:
    os.chdir("/content/deep-actuarial-science")
    sys.path.insert(0, "src")
os.environ.setdefault("DO_NOT_TRACK", "1")
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")

from deep_actuarial_science.notebook import run, load, show, leaderboard, parameters, mlflow_ui
FAST = os.environ.get("DAS_ENV") == "smoke"   # CI runs every notebook in smoke mode


In [ ]:
run("frequency", models=["glm", "fnn"])
leaderboard("frequency", ["glm", "fnn"])

In [ ]:
load("fnn.freq_reconciliation")

## Open the box: the learned brand embedding
Brands close together in this plane have similar effects on frequency.

In [ ]:
import matplotlib.pyplot as plt
fnn = load("fnn.freq_model")
E = fnn.net.emb[0].weight.detach().numpy()
names = fnn.pre.named_transformers_["cats"].categories_[0]
fig, ax = plt.subplots(figsize=(5, 4)); ax.scatter(E[:, 0], E[:, 1])
for n, (a, b) in zip(names, E): ax.annotate(n, (a, b))
ax.set_title("VehBrand embedding"); plt.show()

## Production checks
* **Balance**: `balance_learn` should be ≈ 1 (sum of predictions = sum of claims). Early stopping breaks it; `balance_correction: true` restores it.
* **Stability**: refit with another `seed` and compare predictions.

In [ ]:
show("frequency", "fnn", ["training", "calibration", "marginal_bonusmalus"])
load("fnn.freq_metrics")

## Exercise
Re-run with `frequency.models.fnn.seed` set to 3 other values. How much does the test deviance move? Compare that spread with the gap to the GLM.

## Explore the runs in MLflow
Parameters, metrics, figures and the benchmark table of every run above. In Colab the UI opens below and in a new tab through Colab's proxy (MLflow runs on the Colab machine, so a plain localhost link would point at your own computer).

In [ ]:
mlflow_ui()